In [1]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score, mean_squared_error, confusion_matrix, ConfusionMatrixDisplay

from collections import Counter
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**DATASET PATH**

In [2]:
DATASET_PATH = "/content/drive/MyDrive/datasets"

**CLASS LABELS**

In [3]:
classes = [
    "normal",
    "diabetes",
    "glaucoma",
    "cataract",
    "myopia",
    "hypertension",
    "ageDegeneration",
    "others"
]

label_map = {cls: i for i, cls in enumerate(classes)}

**MISSING DATA HANDLING**

**CHECK EMPTY FOLDERS**

In [5]:
for cls in classes:
    folder = os.path.join(DATASET_PATH, cls)

    if len(os.listdir(folder)) == 0:
        print(f"⚠️ Missing data: {cls} folder is empty")

**DETECT CORRUPTED**

In [ ]:
corrupted_files = []

for cls in classes:
    folder = os.path.join(DATASET_PATH, cls)

    for file in os.listdir(folder):
        path = os.path.join(folder, file)

        img = cv2.imread(path)

        if img is None:
            corrupted_files.append(path)

print("Total corrupted images:", len(corrupted_files))

**Missing Image**

In [ ]:
X = []
y = []

missing_count = 0

for cls in classes:
    folder = os.path.join(DATASET_PATH, cls)

    for file in os.listdir(folder):
        path = os.path.join(folder, file)

        img = cv2.imread(path)

        # 🔴 HANDLE MISSING IMAGE
        if img is None:
            missing_count += 1
            continue

        img = cv2.resize(img, (64, 64))
        img = img / 255.0
        img = img.flatten()

        X.append(img)
        y.append(label_map[cls])

print("Missing/Skipped images:", missing_count)

**Nan Value Check**

In [ ]:
X = np.array(X)
y = np.array(y)

print("NaN in X:", np.isnan(X).sum())
print("NaN in y:", np.isnan(y).sum())

In [ ]:
from collections import Counter

class_counts = Counter(y)

print("📊 Class Distribution:")
for cls, count in class_counts.items():
    print(f"Class {cls}: {count} images")

In [ ]:
# reverse label map
reverse_map = {v:k for k,v in label_map.items()}

print("📊 Class Distribution (Names):")
for cls, count in class_counts.items():
    print(f"{reverse_map[cls]}: {count}")

In [ ]:
import matplotlib.pyplot as plt

labels = [reverse_map[i] for i in class_counts.keys()]
values = list(class_counts.values())

plt.figure()
plt.bar(labels, values)
plt.xticks(rotation=45)
plt.title("Class Distribution")
plt.xlabel("Classes")
plt.ylabel("Number of Images")
plt.show()

In [ ]:
counts = list(class_counts.values())

if max(counts) - min(counts) < 0.1 * max(counts):
    print("✅ Dataset is BALANCED")
else:
    print("⚠️ Dataset is IMBALANCED")

**Fixing Imbalance**

**CHECK BEFORE BALANCING**

In [ ]:
from collections import Counter

print("Before Balancing:")
print(Counter(y))

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

**DEFINE AUGMENTATION**

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,

    # augmentation (IMPORTANT)
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

**TEST DATA**

In [ ]:
test_datagen = ImageDataGenerator(rescale=1./255)

**LOAD DATA FROM FOLDERS**

In [ ]:
train_data = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

**SPLIT TRAIN**

In [ ]:
train_data = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

val_data = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

In [ ]:
validation_split=0.2

In [ ]:
import matplotlib.pyplot as plt

images, labels = next(train_data)

plt.figure(figsize=(10,5))

for i in range(5):
    plt.subplot(1,5,i+1)
    plt.imshow(images[i])
    plt.axis("off")

plt.show()

**IMAGE PREPROCESSING**

In [ ]:
import cv2
import numpy as np

IMAGE_SIZE = 224   # for CNN (VGG16 / ResNet)
def preprocess_image(path):
    img = cv2.imread(path)

    # ❌ corrupted image check
    if img is None:
        return None

    # ✅ resize
    img = cv2.resize(img, (224, 224))

    # ✅ BGR → RGB
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # -------------------------------
    # 🔥 ADVANCED PROCESSING START
    # -------------------------------

    # ✅ Gaussian Blur (noise reduction)
    img = cv2.GaussianBlur(img, (3, 3), 0)

    # ✅ CLAHE (contrast enhancement)
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    l = clahe.apply(l)

    lab = cv2.merge((l, a, b))
    img = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

    # -------------------------------
    # 🔥 ADVANCED PROCESSING END
    # -------------------------------

    # ✅ normalize
    img = img / 255.0

    return img


**LOAD DATASET**

In [ ]:
def data_generator(paths, labels, batch_size=32):
    while True:
        for i in range(0, len(paths), batch_size):
            batch_paths = paths[i:i+batch_size]
            batch_labels = labels[i:i+batch_size]

            X_batch = []
            y_batch = []

            for p, l in zip(batch_paths, batch_labels):
                img = preprocess_image(p)
                if img is not None:
                    X_batch.append(img)
                    y_batch.append(l)

            yield np.array(X_batch), np.array(y_batch)

In [ ]:
import os

DATASET_PATH = "/content/drive/MyDrive/datasets"  # change if needed

classes = os.listdir(DATASET_PATH)

paths = []
labels = []

label_map = {cls: i for i, cls in enumerate(classes)}

for cls in classes:
    folder = os.path.join(DATASET_PATH, cls)

    for file in os.listdir(folder):
        paths.append(os.path.join(folder, file))
        labels.append(label_map[cls])

print("Total images:", len(paths))
print("Classes:", label_map)

In [ ]:
import matplotlib.pyplot as plt

sample_path = paths[0]

original = cv2.imread(sample_path)
original = cv2.cvtColor(original, cv2.COLOR_BGR2RGB)

processed = preprocess_image(sample_path)

plt.figure(figsize=(8,4))

plt.subplot(1,2,1)
plt.imshow(original)
plt.title("Original")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(processed)
plt.title("Processed")
plt.axis("off")

plt.show()

In [ ]:
def data_generator(paths, labels, batch_size=32):
    while True:
        for i in range(0, len(paths), batch_size):
            batch_paths = paths[i:i+batch_size]
            batch_labels = labels[i:i+batch_size]

            X_batch = []
            y_batch = []

            for p, l in zip(batch_paths, batch_labels):
                img = preprocess_image(p)

                if img is not None:
                    X_batch.append(img)
                    y_batch.append(l)   # ✅ FIXED

            yield np.array(X_batch), np.array(y_batch)

**FEATURE SELECTION**

In [ ]:
X = []
y_small = []

LIMIT = 500   # 🔥 DO NOT increase (avoid crash)

count = 0

for p, l in zip(paths, labels):
    img = preprocess_image(p)

    if img is not None:
        X.append(img)
        y_small.append(l)
        count += 1

    if count >= LIMIT:
        break

X = np.array(X)
y_small = np.array(y_small)

print("Loaded subset:", X.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler

# Flatten
X_flat = X.reshape(len(X), -1)

# Normalize
scaler = StandardScaler()
X_flat = scaler.fit_transform(X_flat)

print("Feature shape:", X_flat.shape)

**Filter Method**

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

selector = SelectKBest(score_func=f_classif, k=50)
X_filter = selector.fit_transform(X_flat, y_small)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X_train, X_test, y_train, y_test = train_test_split(
    X_filter, y_small, test_size=0.2, random_state=42
)

model = LogisticRegression(max_iter=200)
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score

filter_acc = accuracy_score(y_test, y_pred)

print("Filter Accuracy:", filter_acc)

**WRAPPER METHOD**

In [ ]:
X        # small image subset (e.g., 300–500 samples)
y_small  # labels

**Flatten image**

In [ ]:
X_flat = X.reshape(len(X), -1)
print("Flatten shape:", X_flat.shape)

**Normalize features**

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_flat = scaler.fit_transform(X_flat)

**PCA**

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=100)   # reduce features
X_pca = pca.fit_transform(X_flat)

print("After PCA:", X_pca.shape)

# **Wrapper Method**

In [ ]:
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=300)

selector = SelectFromModel(model, max_features=50)
X_wrapper = selector.fit_transform(X_pca, y_small)

print("Wrapper selected shape:", X_wrapper.shape)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(
    X_wrapper, y_small, test_size=0.2, random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
wrapper_acc = acc
print("Wrapper Accuracy:", acc)

**EMBEDDED METHOD**

In [ ]:
X        # small subset images
y_small  # labels

In [ ]:
X_flat = X.reshape(len(X), -1)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_flat = scaler.fit_transform(X_flat)

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=100)
X_pca = pca.fit_transform(X_flat)

print("After PCA:", X_pca.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectFromModel

# L1 = automatic feature selection
model = LogisticRegression(penalty='l1', solver='liblinear', max_iter=300)

selector = SelectFromModel(model)
X_embedded = selector.fit_transform(X_pca, y_small)

print("Embedded selected shape:", X_embedded.shape)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(
    X_embedded, y_small, test_size=0.2, random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
embedded_acc = acc
print("Embedded Accuracy:", acc)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.title("Embedded Method - Confusion Matrix")
plt.show()

**Accuracy Comparison**

In [ ]:
import matplotlib.pyplot as plt

methods = ['Filter', 'Wrapper', 'Embedded']
accuracies = [filter_acc, wrapper_acc, embedded_acc]

plt.figure(figsize=(6,4))
plt.bar(methods, accuracies)

for i, v in enumerate(accuracies):
    plt.text(i, v + 0.01, f"{v:.2f}", ha='center')

plt.ylim(0,1)
plt.title("Accuracy Comparison")
plt.show()

# **LINEAR REGRESSION**

**TRAIN MODEL**

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

X_train, X_test, y_train, y_test = train_test_split(
    X_filter, y_small, test_size=0.2, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

**PREDICT**

In [ ]:
y_pred_continuous = model.predict(X_test)

# convert to class labels
y_pred = y_pred_continuous.round().astype(int)

In [ ]:
# keep values in valid class range
y_pred[y_pred < 0] = 0
y_pred[y_pred > len(set(y_small))-1] = len(set(y_small))-1

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)
print("Linear Regression Accuracy:", accuracy)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm)
disp.plot()

# **LOGISTIC REGRESSION**

**TRAIN MODEL**

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# split data
X_train, X_test, y_train, y_test = train_test_split(
    X_wrapper, y_small, test_size=0.2, random_state=42
)

# model
model = LogisticRegression(max_iter=1000)

# train
model.fit(X_train, y_train)

# predict
y_pred = model.predict(X_test)

# accuracy
acc = accuracy_score(y_test, y_pred)
print("Logistic Regression Accuracy:", acc)

# detailed report
print(classification_report(y_test, y_pred))

In [ ]:
import cv2
import numpy as np

class_names = ['cataract','diabetes','glaucoma','normal','others']

def predict_image(path):
    img = cv2.imread(path)

    if img is None:
        print("Image not found")
        return

    # preprocessing (same as training)
    img = cv2.resize(img, (224,224))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = img / 255.0
    img = img.reshape(1, -1)

    # feature selection
    img = selector.transform(img)

    # prediction
    pred = model.predict(img)

    print("Predicted Disease:", class_names[int(pred[0])])

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectFromModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# model with L1 (important for feature selection)
base_model = LogisticRegression(penalty='l1', solver='liblinear', max_iter=1000)

# embedded feature selection
selector = SelectFromModel(base_model, max_features=40)

X_embed = selector.fit_transform(X_flat, y_small)

print("Selected features:", X_embed.shape)

# split
X_train, X_test, y_train, y_test = train_test_split(
    X_embed, y_small, test_size=0.2, random_state=42
)

# train final model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# accuracy
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print("Embedded Method Accuracy:", acc)

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt

class_names = [
    'others',
    'normal',
    'myopia',
    'hypertension',
    'glaucoma',
    'diabetes',
    'cataract',
    'ageDegeneration'
]

folder_path = "/content/drive/MyDrive/datasets"   # FIX THIS PATH

found = False   # to check if anything runs

for class_name in class_names:
    class_folder = os.path.join(folder_path, class_name)

    print("Checking:", class_folder)

    if not os.path.exists(class_folder):
        print("❌ Folder NOT found:", class_folder)
        continue

    files = os.listdir(class_folder)
    print("Files found:", len(files))

    for file in files:
        path = os.path.join(class_folder, file)

        img = cv2.imread(path)

        if img is None:
            print("❌ Image failed:", path)
            continue

        print("✅ Showing:", path)
        found = True

        # display image
        display = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # preprocessing
        img = cv2.resize(img, (224,224))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = img / 255.0
        img = img.reshape(1, -1)

        # feature selection
        img = selector.transform(img)

        # prediction
        pred = model.predict(img)
        pred_class = int(pred[0])
        pred_class = max(0, min(pred_class, len(class_names)-1))

        # show result
        plt.imshow(display)
        plt.title(f"Actual: {class_name} | Predicted: {class_names[pred_class]}")
        plt.axis('off')
        plt.show()

        break   # show one image per class

if not found:
    print("🚨 NOTHING RUN → CHECK PATH OR DATASET")

# **POLYNOMIAL REGRESSION**

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np

# 🔥 Reduce features first (VERY IMPORTANT or it will crash)
from sklearn.decomposition import PCA

pca = PCA(n_components=100)   # reduce 150k → 100
X_pca = pca.fit_transform(X_flat)

# 🔥 Polynomial Features
poly = PolynomialFeatures(degree=2)
X_poly = poly.fit_transform(X_pca)

# split
X_train, X_test, y_train, y_test = train_test_split(
    X_poly, y_small, test_size=0.2, random_state=42
)

# model
model = LinearRegression()
model.fit(X_train, y_train)

# prediction
y_pred = model.predict(X_test)

# convert regression output → class
y_pred_class = np.round(y_pred).astype(int)

# fix bounds
y_pred_class = np.clip(y_pred_class, 0, len(np.unique(y_small))-1)

# accuracy
acc = accuracy_score(y_test, y_pred_class)
print("Polynomial Regression Accuracy:", acc)

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt

class_names = [
    'others','normal','myopia','hypertension',
    'glaucoma','diabetes','cataract','ageDegeneration'
]

folder_path = "/content/drive/MyDrive/datasets"   # change if needed

for class_name in class_names:
    class_folder = os.path.join(folder_path, class_name)

    if not os.path.exists(class_folder):
        continue

    for file in os.listdir(class_folder):
        path = os.path.join(class_folder, file)

        img = cv2.imread(path)
        if img is None:
            continue

        display = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # preprocess
        img = cv2.resize(img, (224,224))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = img / 255.0
        img = img.reshape(1, -1)

        # SAME pipeline
        img = pca.transform(img)
        img = poly.transform(img)

        # predict
        pred = model.predict(img)
        pred_class = int(round(pred[0]))
        pred_class = max(0, min(pred_class, len(class_names)-1))

        # show
        plt.imshow(display)
        plt.title(f"Actual: {class_name} | Predicted: {class_names[pred_class]}")
        plt.axis('off')
        plt.show()

        break   # one per class

**NEURAL NETWORK**

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# -----------------------------
# Prepare Data
# -----------------------------
X = X_flat   # your flattened images
y = y_small

# normalize labels (0 to n-1)
num_classes = len(np.unique(y))

# one-hot encoding
def one_hot(y, c):
    return np.eye(c)[y]

y_encoded = one_hot(y, num_classes)

# split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42
)

# -----------------------------
# Network Architecture
# -----------------------------
input_size = X_train.shape[1]
hidden_size = 64
output_size = num_classes

# weights initialization
np.random.seed(42)
W1 = np.random.randn(input_size, hidden_size) * 0.01
b1 = np.zeros((1, hidden_size))

W2 = np.random.randn(hidden_size, output_size) * 0.01
b2 = np.zeros((1, output_size))

# -----------------------------
# Activation Functions
# -----------------------------
def relu(Z):
    return np.maximum(0, Z)

def relu_deriv(Z):
    return Z > 0

def softmax(Z):
    expZ = np.exp(Z - np.max(Z, axis=1, keepdims=True))
    return expZ / np.sum(expZ, axis=1, keepdims=True)

# -----------------------------
# Training Parameters
# -----------------------------
lr = 0.01
epochs = 50
lambda_reg = 0.01   # L2 regularization

# -----------------------------
# TRAINING LOOP
# -----------------------------
for epoch in range(epochs):

    # ===== FORWARD PROP =====
    Z1 = np.dot(X_train, W1) + b1
    A1 = relu(Z1)

    Z2 = np.dot(A1, W2) + b2
    A2 = softmax(Z2)

    # ===== LOSS (Cross Entropy + L2) =====
    m = X_train.shape[0]

    log_loss = -np.sum(y_train * np.log(A2 + 1e-8)) / m
    l2_loss = (lambda_reg / (2*m)) * (np.sum(W1**2) + np.sum(W2**2))

    loss = log_loss + l2_loss

    # ===== BACKPROP =====
    dZ2 = A2 - y_train
    dW2 = np.dot(A1.T, dZ2)/m + (lambda_reg/m)*W2
    db2 = np.sum(dZ2, axis=0, keepdims=True)/m

    dA1 = np.dot(dZ2, W2.T)
    dZ1 = dA1 * relu_deriv(Z1)
    dW1 = np.dot(X_train.T, dZ1)/m + (lambda_reg/m)*W1
    db1 = np.sum(dZ1, axis=0, keepdims=True)/m

    # ===== UPDATE =====
    W1 -= lr * dW1
    b1 -= lr * db1
    W2 -= lr * dW2
    b2 -= lr * db2

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss:.4f}")

# -----------------------------
# TEST ACCURACY
# -----------------------------
Z1 = np.dot(X_test, W1) + b1
A1 = relu(Z1)
Z2 = np.dot(A1, W2) + b2
A2 = softmax(Z2)

y_pred = np.argmax(A2, axis=1)
y_true = np.argmax(y_test, axis=1)

acc = accuracy_score(y_true, y_pred)
print("Neural Network Accuracy:", acc)

# **CNN MODEL**

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
import matplotlib.pyplot as plt

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,

    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

In [ ]:
DATASET_PATH = "/content/drive/MyDrive/datasets"   # fix if needed

train_data = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(128,128),   # ✅ MUST BE 128
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

val_data = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(128,128),   # ✅ MUST BE 128
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(16, (3,3), activation='relu', input_shape=(128,128,3)),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Conv2D(32, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.5),

    tf.keras.layers.Dense(train_data.num_classes, activation='softmax')
])

In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    steps_per_epoch=50,        # 🔥 ADD THIS
    validation_steps=20        # 🔥 ADD THIS
)

In [ ]:
cnn_loss, cnn_acc = model.evaluate(val_data)

print("CNN Accuracy:", cnn_acc)

In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title("Model Accuracy")
plt.legend(["Train","Validation"])
plt.show()

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

class_names = list(train_data.class_indices.keys())

folder_path = "/content/drive/MyDrive/datasets"

for class_name in class_names:

    class_folder = os.path.join(folder_path, class_name)

    if not os.path.exists(class_folder):
        continue

    count = 0

    for file in os.listdir(class_folder):

        path = os.path.join(class_folder, file)

        img = cv2.imread(path)
        if img is None:
            continue

        # display image
        display = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # ✅ FIXED SIZE (IMPORTANT)
        img = cv2.resize(img, (128,128))
        img = img / 255.0
        img = np.expand_dims(img, axis=0)

        # prediction
        pred = model.predict(img, verbose=0)
        pred_class = np.argmax(pred)

        # show
        plt.imshow(display)
        plt.title(f"Actual: {class_name} | Predicted: {class_names[pred_class]}")
        plt.axis('off')
        plt.show()

        count += 1
        if count == 2:
            break

In [ ]:
import matplotlib.pyplot as plt

# 🔥 PUT YOUR REAL VALUES HERE
linear_acc = 0.60
logistic_acc = 0.75
poly_acc = 0.65
cnn_acc = 0.49   # your result

models = ['Linear', 'Logistic', 'Polynomial', 'CNN']
accuracies = [linear_acc, logistic_acc, poly_acc, cnn_acc]

plt.figure(figsize=(7,5))
bars = plt.bar(models, accuracies)

plt.title("Accuracy Comparison of Models")
plt.xlabel("Models")
plt.ylabel("Accuracy")

# values on bars
for i, v in enumerate(accuracies):
    plt.text(i, v + 0.01, f"{v:.2f}", ha='center')

plt.ylim(0,1)
plt.show()

# **VGG16**

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np
import cv2, os
import matplotlib.pyplot as plt

In [ ]:
DATASET_PATH = "/content/drive/MyDrive/datasets"   # 🔥 fix if needed

train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

train_data = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(128,128),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

val_data = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(128,128),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

class_names = list(train_data.class_indices.keys())
print(class_names)

In [ ]:
base_model = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(128,128,3)
)

# freeze base layers
for layer in base_model.layers:
    layer.trainable = False

model = Sequential([
    base_model,
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),          # ✅ regularization
    Dense(len(class_names), activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

In [ ]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    steps_per_epoch=50,        # 🔥 reduce from 160 → 50
    validation_steps=20        # 🔥 reduce validation
)

In [ ]:
# Final Training Accuracy
train_acc = history.history['accuracy'][-1]

# Final Validation Accuracy
val_acc = history.history['val_accuracy'][-1]

print(f"Final Training Accuracy: {train_acc*100:.2f}%")
print(f"Final Validation Accuracy: {val_acc*100:.2f}%")

In [ ]:
import matplotlib.pyplot as plt

# Accuracy values
train_acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

epochs = range(1, len(train_acc) + 1)

plt.figure()
plt.plot(epochs, train_acc, label='Training Accuracy')
plt.plot(epochs, val_acc, label='Validation Accuracy')

plt.title('Training vs Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import cv2
import os

test_path = "/content/drive/MyDrive/datasets"

for class_name in class_names:
    class_folder = os.path.join(test_path, class_name)

    if not os.path.exists(class_folder):
        continue

    count = 0

    for file in os.listdir(class_folder):
        path = os.path.join(class_folder, file)

        img = cv2.imread(path)
        if img is None:
            continue

        display = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        img = cv2.resize(img, (128,128))
        img = img / 255.0
        img = np.expand_dims(img, axis=0)

        pred = model.predict(img, verbose=0)
        pred_class = np.argmax(pred)

        plt.imshow(display)
        plt.title(f"Actual: {class_name} | Pred: {class_names[pred_class]}")
        plt.axis('off')
        plt.show()

        count += 1
        if count == 2:
            break

# **MobileNetV2**

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

In [ ]:
DATASET_PATH = '/content/drive/MyDrive/datasets'

train_data = train_datagen.flow_from_directory(
    DATASET_PATH,   # ✅ correct
    target_size=(160,160),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

val_data = train_datagen.flow_from_directory(
    DATASET_PATH,   # ✅ correct
    target_size=(160,160),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

In [ ]:
class_names = list(train_data.class_indices.keys())
print(class_names)

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout

base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(160,160,3)
)

# Freeze base layers
for layer in base_model.layers:
    layer.trainable = False

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(len(class_names), activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    steps_per_epoch=50,
    validation_steps=20
)

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

test_datagen = ImageDataGenerator(rescale=1./255)

test_data = test_datagen.flow_from_directory(
    '/content/drive/MyDrive/datasets',  # 🔥 change if needed
    target_size=(160,160),   # ✅ must match your model
    batch_size=32,
    class_mode='categorical',
    shuffle=False   # ⚠️ VERY IMPORTANT for evaluation
)

In [ ]:
test_loss, test_acc = model.evaluate(test_data)

print(f"Test Accuracy: {test_acc*100:.2f}%")
print(f"Test Loss: {test_loss:.4f}")

In [ ]:
import matplotlib.pyplot as plt

# Accuracy
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model Accuracy')
plt.legend(['Train', 'Validation'])
plt.show()

# Loss
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model Loss')
plt.legend(['Train', 'Validation'])
plt.show()

In [ ]:
import cv2
import matplotlib.pyplot as plt
import os

test_path = '/content/drive/MyDrive/datasets'

for class_name in class_names:
    class_folder = os.path.join(test_path, class_name)

    if not os.path.exists(class_folder):
        continue

    count = 0

    for file in os.listdir(class_folder):
        path = os.path.join(class_folder, file)

        img = cv2.imread(path)
        if img is None:
            continue

        display = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        img = cv2.resize(img, (160,160))  # ✅ match MobileNet
        img = img / 255.0
        img = np.expand_dims(img, axis=0)

        pred = model.predict(img, verbose=0)
        pred_class = np.argmax(pred)

        plt.imshow(display)
        plt.title(f"Actual: {class_name} | Pred: {class_names[pred_class]}")
        plt.axis('off')
        plt.show()

        count += 1
        if count == 2:
            break

**UNFREEZE TOP LAYERS (Fine-tuning)**

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# Load base model
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(160,160,3)
)

# 🔥 Unfreeze top layers (important)
for layer in base_model.layers[:-30]:   # freeze early layers
    layer.trainable = False

for layer in base_model.layers[-30:]:   # unfreeze last 30 layers
    layer.trainable = True

# Build model
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(256, activation='relu'),   # 🔥 increased capacity
    Dropout(0.5),
    Dense(len(class_names), activation='softmax')
])

**COMPILE WITH LOWER LEARNING RATE**

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=0.0001),  # 🔥 lower LR for fine-tuning
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

**ADD CALLBACKS**

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=3,
    min_lr=1e-6
)

**TRAIN LONGER**

In [ ]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=25,   # 🔥 increased
    steps_per_epoch=50,
    validation_steps=20,
    callbacks=[early_stop, reduce_lr]
)

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

test_datagen = ImageDataGenerator(rescale=1./255)

test_data = test_datagen.flow_from_directory(
    '/content/drive/MyDrive/datasets',  # 🔥 update if needed
    target_size=(160,160),   # MUST match your model
    batch_size=32,
    class_mode='categorical',
    shuffle=False   # ⚠️ keep False
)

In [ ]:
test_loss, test_acc = model.evaluate(test_data)

print("Test Accuracy:", round(test_acc*100, 2), "%")
print("Test Loss:", round(test_loss, 4))

In [ ]:
import cv2
import matplotlib.pyplot as plt
import os

test_path = '/content/drive/MyDrive/datasets'

for class_name in class_names:
    class_folder = os.path.join(test_path, class_name)

    if not os.path.exists(class_folder):
        continue

    count = 0

    for file in os.listdir(class_folder):
        path = os.path.join(class_folder, file)

        img = cv2.imread(path)
        if img is None:
            continue

        display = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        img = cv2.resize(img, (160,160))
        img = img / 255.0
        img = np.expand_dims(img, axis=0)

        pred = model.predict(img, verbose=0)
        pred_class = np.argmax(pred)

        plt.imshow(display)
        plt.title(f"Actual: {class_name} | Pred: {class_names[pred_class]}")
        plt.axis('off')
        plt.show()

        count += 1
        if count == 2:
            break

**Generalization**

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,

    rotation_range=25,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.2,
    shear_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

In [ ]:
Dense(256, activation='relu'),
Dropout(0.5),   # 🔥 prevents overfitting

In [ ]:
from tensorflow.keras.regularizers import l2

In [ ]:
Dense(256, activation='relu', kernel_regularizer=l2(0.001)),
Dropout(0.5),

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

In [ ]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=25,
    callbacks=[early_stop]
)

In [ ]:
from tensorflow.keras.callbacks import ReduceLROnPlateau

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=3,
    min_lr=1e-6
)

In [ ]:
callbacks=[early_stop, reduce_lr]

In [ ]:
train_acc = history.history['accuracy'][-1]
val_acc = history.history['val_accuracy'][-1]

print("Training Accuracy:", train_acc)
print("Validation Accuracy:", val_acc)

gap = train_acc - val_acc
print("Generalization Gap:", gap)